In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import re
import sys
import matplotlib.pyplot as plt
from functools import reduce
import glob, os

In [2]:
RUTA = "Valuaciones_USA"
# Excluye archivos temporales de Excel (prefijo ~$)
files = sorted([f for f in glob.glob(f"{RUTA}/*_score_ponderados.xlsx") if not os.path.basename(f).startswith("~")])

registros = []
for f in files:
    ticker = os.path.basename(f).split("_score_ponderados")[0]
    df = pd.read_excel(f, index_col=0)

    # Columnas con formato MesAño (ej. Dec2024, Aug2025)
    cols_periodo = [c for c in df.columns if isinstance(c, str) and re.match(r"^[A-Za-z]{3}\d{4}$", c)]

    # Últimos 3 disponibles
    ultimos_3 = cols_periodo[-3:]

    # Fila Score Total
    score_row = df[df.index.astype(str).str.contains("Score Total", case=False, na=False)]

    for col in ultimos_3:
        valor = score_row[col].values[0] if not score_row.empty else None
        mes = col[:3]
        anio = int(col[3:])
        registros.append({"Ticker": ticker, "Periodo": col, "Mes": mes, "Año": anio, "Score_Total": valor})

df_scores = pd.DataFrame(registros)
df_scores["Score_Total"] = pd.to_numeric(df_scores["Score_Total"], errors="coerce").round(2)


In [3]:
# Último año disponible por empresa
ultimo_anio = df_scores.groupby("Ticker")["Año"].max().reset_index()
ultimo_score = df_scores.merge(ultimo_anio, on=["Ticker", "Año"])

# Empresas con score > 60 en su último año
tickers_validos = ultimo_score[ultimo_score["Score_Total"] > 60]["Ticker"].tolist()

df_scores_filtrado = df_scores[df_scores["Ticker"].isin(tickers_validos)].reset_index(drop=True)

print(f"Empresas que pasan el filtro: {len(tickers_validos)}")
print(sorted(tickers_validos))
df_scores_filtrado

Empresas que pasan el filtro: 17
['ACN', 'ADBE', 'AMZN', 'CRM', 'GOOG', 'INTU', 'MA', 'MELI', 'META', 'MNST', 'MSFT', 'NFLX', 'NKE', 'PCAR', 'TER', 'TSM', 'V']


,Ticker,Periodo,Mes,Año,Score_Total
0,ACN,Aug2023,Aug,2023,77.00
1,ACN,Aug2024,Aug,2024,70.50
2,ACN,Aug2025,Aug,2025,70.25
3,ADBE,Nov2021,Nov,2021,81.50
4,ADBE,Nov2022,Nov,2022,83.75
5,ADBE,Nov2023,Nov,2023,82.50
6,AMZN,Dec2022,Dec,2022,25.50
7,AMZN,Dec2023,Dec,2023,42.25
8,AMZN,Dec2024,Dec,2024,62.00
9,CRM,Jan2023,Jan,2023,45.00


In [4]:
dfs_hits = []
sin_hits = []

for ticker in tickers_validos:
    f = f"{RUTA}/{ticker}_valuaciones_hits_por_anio.xlsx"
    if not os.path.exists(f):
        sin_hits.append(ticker)
        continue

    df = pd.read_excel(f)

    if df.empty:
        sin_hits.append(ticker)
        continue

    df.insert(0, "Ticker", ticker)
    # Normaliza hit a float (True->1.0, False->0.0, NaN->NaN)
    df["hit"] = pd.to_numeric(df["hit"].map({True: 1, False: 0}).fillna(df["hit"]), errors="coerce")
    dfs_hits.append(df)

df_hits = pd.concat(dfs_hits, ignore_index=True)
df_hits.columns = df_hits.columns.str.strip()

print(f"Sin datos de valuación: {sin_hits}")
print(f"\ndf_hits shape: {df_hits.shape}")
df_hits.head(10)

Sin datos de valuación: ['NKE', 'TER']

df_hits shape: (590, 9)


,Ticker,Año,Metodo,Valor,hit,tth_weeks,hit_date,side,Mes
0,ACN,2011,GF_Value_Guru,49.88,1.0,7.0,2011-10-24,long,8
1,ACN,2012,GF_Value_Guru,60.78,1.0,21.0,2013-01-28,long,8
2,ACN,2013,GF_Value_Guru,69.68,1.0,16.0,2013-12-23,long,8
3,ACN,2014,GF_Value_Guru,79.16,1.0,15.0,2014-12-15,long,8
4,ACN,2015,GF_Value_Guru,88.65,1.0,1.0,2015-09-14,long,8
5,ACN,2016,GF_Value_Guru,101.19,1.0,0.0,2016-09-05,at_target,8
6,ACN,2017,GF_Value_Guru,120.00,1.0,0.0,2017-09-04,at_target,8
7,ACN,2018,GF_Value_Guru,143.79,1.0,0.0,2018-09-03,at_target,8
8,ACN,2019,GF_Value_Guru,171.64,1.0,0.0,2019-09-02,at_target,8
9,ACN,2020,GF_Value_Guru,189.14,1.0,2.0,2020-09-21,short,8


In [11]:
# Últimos 6 años disponibles por empresa
anio_max = df_hits.groupby("Ticker")["Año"].max()
df_hits_6 = df_hits[df_hits.apply(lambda r: r["Año"] >= anio_max[r["Ticker"]] - 5, axis=1)].copy()

# Solo filas donde hit tiene valor (excluye NaN = valuación en 0)
df_validos = df_hits_6[df_hits_6["hit"].notna()].copy()

# Mes numérico -> abreviatura (ej. 8 -> "Aug")
df_validos["Mes"] = pd.to_datetime(df_validos["Mes"].astype(int), format="%m").dt.strftime("%b")

# % aciertos por empresa y método
resumen = (
    df_validos
    .groupby(["Ticker", "Metodo"])
    .agg(
        total_años=("hit", "count"),
        hits=("hit", "sum"),
        años=("Año", lambda x: sorted(x.tolist())),
        años_sin_hit=("Año", lambda x: sorted(x[df_validos.loc[x.index, "hit"] == 0].tolist())),
        mes=("Mes", "first"),
    )
    .assign(pct_hit=lambda x: (x["hits"] / x["total_años"] * 100).round(1))
    .reset_index()
    .sort_values(["Ticker", "pct_hit"], ascending=[True, False])
)

# Solo métodos con los 6 años completos
resumen = resumen[resumen["total_años"] == 6].reset_index(drop=True)

print(resumen.shape)
resumen

(39, 8)


,Ticker,Metodo,total_años,hits,años,años_sin_hit,mes,pct_hit
0,ACN,GF_Value_Guru,6,5.0,"[2020, 2021, 2022, 2023, 2024, 2025]",[2025],Aug,83.3
1,ACN,Peter_Lynch_Value_2,6,5.0,"[2020, 2021, 2022, 2023, 2024, 2025]",[2025],Aug,83.3
2,ADBE,Peter_Lynch_Value_2,6,6.0,"[2018, 2019, 2020, 2021, 2022, 2023]",[],Nov,100.0
3,AMZN,GF_Value_Guru,6,6.0,"[2019, 2020, 2021, 2022, 2023, 2024]",[],Dec,100.0
4,AMZN,Median_PS_Value_Guru,6,5.0,"[2019, 2020, 2021, 2022, 2023, 2024]",[2019],Dec,83.3
5,AMZN,Peter_Lynch_Value_2,6,4.0,"[2019, 2020, 2021, 2022, 2023, 2024]","[2023, 2024]",Dec,66.7
6,CRM,GF_Value_Guru,6,6.0,"[2020, 2021, 2022, 2023, 2024, 2025]",[],Jan,100.0
7,CRM,Median_PS_Value_Guru,6,6.0,"[2020, 2021, 2022, 2023, 2024, 2025]",[],Jan,100.0
8,CRM,Median_PS_Value_Promedio,6,6.0,"[2020, 2021, 2022, 2023, 2024, 2025]",[],Jan,100.0
9,CRM,Median_PS_Value_Promedio_Mediana,6,6.0,"[2020, 2021, 2022, 2023, 2024, 2025]",[],Jan,100.0
